In [3]:
# 安装下huggingface的transformer库和 datasets库

!pip install datasets

Looking in indexes: https://mirrors.aliyun.com/pypi/simple/
     ---------------------------------------- 0.0/26.2 MB ? eta -:--:--
     -------- ------------------------------- 5.5/26.2 MB 27.9 MB/s eta 0:00:01
     -------------- ------------------------- 9.7/26.2 MB 23.2 MB/s eta 0:00:01
     ---------------------- ---------------- 14.9/26.2 MB 23.5 MB/s eta 0:00:01
     ------------------------------ -------- 20.4/26.2 MB 24.4 MB/s eta 0:00:01
     --------------------------------------  26.2/26.2 MB 24.8 MB/s eta 0:00:01
     ---------------------------------------- 26.2/26.2 MB 23.7 MB/s  0:00:01

   -- -------------------------------------  1/18 [pyarrow]
   -- -------------------------------------  1/18 [pyarrow]
   -- -------------------------------------  1/18 [pyarrow]
   -- -------------------------------------  1/18 [pyarrow]
   -- -------------------------------------  1/18 [pyarrow]
   -- -------------------------------------  1/18 [pyarrow]
   -- -----------------------

In [5]:
from datasets import *


e:\anaconda3\envs\pytorch\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [ ]:
# 加载数据集
dataset = load_dataset("madao33/new-title-chinese")
dataset

DatasetDict({
    train: Dataset({
        features: ['title', 'content'],
        num_rows: 5850
    })
    validation: Dataset({
        features: ['title', 'content'],
        num_rows: 1679
    })
})

In [12]:
# 只要训练集
dataset = load_dataset("madao33/new-title-chinese", split="train")
dataset

Dataset({
    features: ['title', 'content'],
    num_rows: 5850
})

# 在dataset中处理数据

这里做的是一个content 总结出对应title的任务

In [14]:
from transformers import AutoTokenizer
tokenizer = AutoTokenizer.from_pretrained("bert-base-chinese")

def process_datset_func(examples):
    model_inputs = tokenizer(examples["content"], max_length=128, truncation=True)
    labels = tokenizer(examples["title"], max_length=32, truncation=True)
    model_inputs["labels"] = labels["input_ids"]
    return model_inputs

proessed_dataset = dataset.map(process_datset_func)
proessed_dataset


Dataset({
    features: ['title', 'content', 'input_ids', 'token_type_ids', 'attention_mask', 'labels'],
    num_rows: 5850
})

当数据量很大， 可以使用批处理的功能， 使用一个batched参数就行

In [15]:
proessed_dataset = dataset.map(process_datset_func, batched=True)
proessed_dataset


Map: 100%|██████████| 5850/5850 [00:03<00:00, 1642.94 examples/s]


Dataset({
    features: ['title', 'content', 'input_ids', 'token_type_ids', 'attention_mask', 'labels'],
    num_rows: 5850
})

还可以指定多进程， 但是子进程和主进程的资源不共享， 因此要吧tokenizer传入进去

In [16]:
proessed_dataset = dataset.map(process_datset_func, batched=True, num_proc=4)
proessed_dataset

Map (num_proc=4):   0%|          | 0/5850 [00:10<?, ? examples/s]


NameError: name 'tokenizer' is not defined

In [ ]:
# 把tokenzier传入到函数里面，方便子进程调用
def process_datset_func(examples, tokenizer=tokenizer):
    model_inputs = tokenizer(examples["content"], max_length=128, truncation=True)
    labels = tokenizer(examples["title"], max_length=32, truncation=True)
    model_inputs["labels"] = labels["input_ids"]
    return model_inputs

proessed_dataset = dataset.map(process_datset_func, batched=True, num_proc=4)


Map (num_proc=4): 100%|██████████| 5850/5850 [00:25<00:00, 231.77 examples/s]


# 去掉dataset不想要的字段

可以使用remove_columns这个参数

In [20]:
processed_dataset = dataset.map(process_datset_func, batched=True, remove_columns=dataset.data.column_names)
processed_dataset

Dataset({
    features: ['input_ids', 'token_type_ids', 'attention_mask', 'labels'],
    num_rows: 5850
})

# 保存和加载数据集

In [21]:
processed_dataset.save_to_disk("./processed_dataset")
processed_dataset = load_from_disk("./processed_dataset")
processed_dataset 

Saving the dataset (1/1 shards): 100%|██████████| 5850/5850 [00:00<00:00, 278832.23 examples/s]


Dataset({
    features: ['input_ids', 'token_type_ids', 'attention_mask', 'labels'],
    num_rows: 5850
})

# 加载其他类型的数据集 


In [26]:
# 加载csv格式的数据集

dataset = load_dataset("csv", data_files="./ChnSentiCorp_htl_all.csv")
dataset 

DatasetDict({
    train: Dataset({
        features: ['label', 'review'],
        num_rows: 7766
    })
})

In [28]:
# 使用from_csv 也可以
dataset = Dataset.from_csv("./ChnSentiCorp_htl_all.csv")
dataset 

Generating train split: 7766 examples [00:00, 50768.81 examples/s]


Dataset({
    features: ['label', 'review'],
    num_rows: 7766
})

# 完整处理数据过程


In [37]:
from transformers import  DataCollatorWithPadding
from torch.utils.data import DataLoader

In [30]:
dataset = load_dataset("csv", data_files="./ChnSentiCorp_htl_all.csv", split='train')
dataset = dataset.filter(lambda x: x["review"] is not None)
dataset

Generating train split: 7766 examples [00:00, 38212.57 examples/s]
Filter: 100%|██████████| 7766/7766 [00:00<00:00, 93186.76 examples/s]


Dataset({
    features: ['label', 'review'],
    num_rows: 7765
})

In [31]:
def process_function(examples):
    tokenized_examples = tokenizer(examples["review"], max_length=128, truncation=True)
    tokenized_examples["labels"] = examples["label"]
    return tokenized_examples

In [32]:
tokenized_dataset = dataset.map(process_function, batched=True, remove_columns=dataset.column_names)
tokenized_dataset

Map: 100%|██████████| 7765/7765 [00:02<00:00, 3649.07 examples/s]


Dataset({
    features: ['input_ids', 'token_type_ids', 'attention_mask', 'labels'],
    num_rows: 7765
})

In [33]:
print(tokenized_dataset[:3])

{'input_ids': [[101, 6655, 4895, 2335, 3763, 1062, 6662, 6772, 6818, 117, 852, 3221, 1062, 769, 2900, 4850, 679, 2190, 117, 1963, 3362, 3221, 107, 5918, 7355, 5296, 107, 4638, 6413, 117, 833, 7478, 2382, 7937, 4172, 119, 2456, 6379, 4500, 1166, 4638, 6662, 5296, 119, 2791, 7313, 6772, 711, 5042, 1296, 119, 102], [101, 1555, 1218, 1920, 2414, 2791, 8024, 2791, 7313, 2523, 1920, 8024, 2414, 3300, 100, 2160, 8024, 3146, 860, 2697, 6230, 5307, 3845, 2141, 2669, 679, 7231, 106, 102], [101, 3193, 7623, 1922, 2345, 8024, 3187, 6389, 1343, 1914, 2208, 782, 8024, 6929, 6804, 738, 679, 1217, 7608, 1501, 4638, 511, 6983, 2421, 2418, 6421, 7028, 6228, 671, 678, 6821, 702, 7309, 7579, 749, 511, 2791, 7313, 3315, 6716, 2523, 1962, 511, 102]], 'token_type_ids': [[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,

In [48]:
# 官方的只能对四个字段进行padding，如果我们有更多的字段，就需要自己写一个collator了， 
# 他们分别是input_ids, attention_mask, token_type_ids, labels
collator = DataCollatorWithPadding(tokenizer=tokenizer, max_length=156, padding="max_length", return_tensors="pt")

In [49]:
dl = DataLoader(tokenized_dataset, batch_size=4, collate_fn=collator, shuffle=True)

In [50]:
num = 0
for batch in dl:
    for k, v in batch.items():
        print(k, v.shape)
    break

input_ids torch.Size([4, 156])
token_type_ids torch.Size([4, 156])
attention_mask torch.Size([4, 156])
labels torch.Size([4])
